# Phishing Detection: Evasion Attacks and Defenses
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sjcALane/llm-security-labs/blob/main/notebooks/04_evasion_attacks_and_defenses.ipynb)

An attacker sends a phishing email that a person reads normally but the model sees differently. We try four attacks on the fine-tuned model and two defenses.

In [ ]:
# In Colab: Runtime > Change runtime type > T4 GPU. Then run this cell.
try:
    import seclab
except ImportError:
    !pip -q install git+https://github.com/sjcALane/llm-security-labs
    import seclab

# Optional: keep saved adapters between Colab sessions.
# from google.colab import drive
# drive.mount("/content/drive")

from seclab import attacks, config, data, defenses, lora, metrics, notebook, prompts
from seclab.model import load, score
notebook.banner()

In [ ]:
df = data.load_emails()
train, val, test = data.split(df, seed=config.SEED)
test_small = data.sample(test, config.N_TEST, seed=config.SEED)
y = test_small["label"].to_numpy()
emails = test_small["text"].tolist()
print(f"{len(train)} training emails, {len(test_small)} test emails")

If you saved the adapter in the LoRA lab, the next cell loads it. If not, it trains a new one.

In [ ]:
lm, tok = load(config.MODEL_NAME, config.device())
train_small = data.sample(train, config.N_TRAIN, seed=config.SEED)
ft = lora.load_or_train(
    lm, tok, train_small, notebook.workdir() / "adapter-clean",
    batch_size=config.BATCH, micro_batch=config.MICRO_BATCH,
)

We only attack phishing emails, since the attacker wants phishing to get through.

In [ ]:
phish = test_small[test_small["label"] == 1]["text"].tolist()
y_phish = [1] * len(phish)
p_clean = score(ft, tok, phish)
print(f"{len(phish)} phishing emails, model catches {(p_clean >= 0.5).mean():.1%}")

## Four attacks

In [ ]:
e = phish[0][:200]
print("original:\n", e, "\n")
print("homoglyph:\n", attacks.homoglyph(e), "\n")
print("zero-width:\n", attacks.zero_width(e), "\n")
print("benign padding:\n", attacks.pad_benign(e), "\n")
print("prompt injection:\n", attacks.inject(e))

The first two look identical to the original. `print` shows text the way a reader sees it. `repr` shows the characters that are really there.

In [ ]:
print(repr(attacks.zero_width(e)[:60]))

## What the model sees

In [ ]:
word = "password"
for w in [word, attacks.homoglyph(word, rate=1.0), attacks.zero_width(word, every=2)]:
    ids = tok.encode(w, add_special_tokens=False)
    print(f"{w!r:40} {len(ids)} tokens: {[tok.decode(i) for i in ids]}")

print("a".encode("utf-8"), "\u0430".encode("utf-8"))

**Question 1.** The Latin `a` is one byte in UTF-8 and the Cyrillic `а` is two. Use that and the token lists above to explain why one swapped letter changes so much for the model.

**Your answer:**

## Measure each attack

In [ ]:
attacked = {
    "homoglyph": [attacks.homoglyph(t, seed=i) for i, t in enumerate(phish)],
    "zero-width": [attacks.zero_width(t) for t in phish],
    "benign padding": [attacks.pad_benign(t, seed=i) for i, t in enumerate(phish)],
    "prompt injection": [attacks.inject(t) for t in phish],
}
asr = {}
for name, texts in attacked.items():
    asr[name] = metrics.attack_success_rate(y_phish, p_clean, score(ft, tok, texts))
    print(f"{name:18} attack success rate {asr[name]:.3f}")

## Defense 1: normalize the input

`defenses.normalize` applies Unicode NFKC normalization, deletes invisible characters, and maps look-alike letters back to Latin.

In [ ]:
print(defenses.normalize(attacks.homoglyph(e)) == e)

asr_norm = {}
for name, texts in attacked.items():
    p = score(ft, tok, [defenses.normalize(t) for t in texts])
    asr_norm[name] = metrics.attack_success_rate(y_phish, p_clean, p)
    print(f"{name:18} {asr[name]:.3f} -> {asr_norm[name]:.3f}")

**Question 2.** Normalization stops two of the attacks and does little against the other two. Which ones, and why?

**Your answer:**

**Question 3.** `normalize` turns Cyrillic letters into Latin ones. Whose legitimate email does that damage? Suggest a better rule.

**Your answer:**

## Defense 2: a hardened system prompt

In [ ]:
print(prompts.SYSTEM_HARDENED)

p_clean_h = score(ft, tok, phish, system=prompts.SYSTEM_HARDENED)
p_inj_h = score(ft, tok, attacked["prompt injection"], system=prompts.SYSTEM_HARDENED)
asr_h = metrics.attack_success_rate(y_phish, p_clean_h, p_inj_h)
print(f"prompt injection: {asr['prompt injection']:.3f} with the plain prompt, {asr_h:.3f} hardened")

**Question 4.** The model was fine-tuned with the plain system prompt and tested here with a different one. Is that a fair test of the hardened prompt? What would make it fair?

**Your answer:**

**Question 5.** Compare the injection success rate here with the zero-shot model in the prompt injection lab. Why would training on labeled emails change how easily the model is injected?

**Your answer:**

In [ ]:
notebook.save_result("Fine-tuned ASR", asr)
notebook.save_result("Fine-tuned ASR, normalized", asr_norm)
notebook.save_result("Fine-tuned ASR, injection, hardened prompt", asr_h)